In [ ]:
RUN_MODE = "databricks"  # "databricks" or "local"


In [ ]:
import os
from pathlib import Path

import yaml


def _resolve_repo_root() -> Path:
    try:
        nb_path = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        if "/notebooks/" in nb_path:
            return Path(nb_path.split("/notebooks/")[0])
    except Exception:
        pass

    cwd = Path.cwd()
    return cwd.parent if cwd.name == "notebooks" else cwd


def _load_config() -> dict:
    root = _resolve_repo_root()
    candidates = [
        root / "config" / "databricks_connect.yaml",
        root / "config" / "databricks_connect.example.yaml",
        Path("config/databricks_connect.yaml"),
        Path("../config/databricks_connect.yaml"),
    ]
    for p in candidates:
        if p.exists():
            with p.open("r", encoding="utf-8") as f:
                cfg = yaml.safe_load(f) or {}
            cfg["_config_path"] = str(p)
            return cfg
    raise FileNotFoundError("Could not find config/databricks_connect.yaml")


def _show_df(df, limit: int = 20) -> None:
    try:
        display(df.limit(limit))
    except NameError:
        df.show(limit, truncate=False)


In [ ]:
cfg = _load_config()
run_mode = str(RUN_MODE).strip().lower()
if run_mode not in {"databricks", "local"}:
    raise ValueError("RUN_MODE must be 'databricks' or 'local'")

if run_mode == "local":
    local_cfg = cfg.get("local_connect") or {}
    profile = str(local_cfg.get("profile", "dbx"))
    cli_path = Path(str(local_cfg.get("cli_path", ".tools/databricks-cli/databricks.exe")))
    compute_mode = str(local_cfg.get("compute_mode", "cluster")).lower()

    root = _resolve_repo_root()
    if not cli_path.is_absolute():
        cli_path = (root / cli_path).resolve()

    os.environ["DATABRICKS_CONFIG_PROFILE"] = profile
    os.environ["DATABRICKS_CLI_PATH"] = str(cli_path)

    if compute_mode == "cluster":
        cluster_id = str(local_cfg.get("cluster_id", "")).strip()
        if not cluster_id:
            raise ValueError("local_connect.cluster_id is required for compute_mode='cluster'")
        os.environ["DATABRICKS_CLUSTER_ID"] = cluster_id
        os.environ.pop("DATABRICKS_SERVERLESS_COMPUTE_ID", None)
    else:
        os.environ["DATABRICKS_SERVERLESS_COMPUTE_ID"] = str(
            local_cfg.get("serverless_compute_id", "auto")
        )
        os.environ.pop("DATABRICKS_CLUSTER_ID", None)

    from databricks.connect import DatabricksSession

    spark = DatabricksSession.builder.getOrCreate()
    print(f"Local mode enabled with profile={profile}")
else:
    if "spark" not in globals():
        raise RuntimeError("In Databricks mode, attach compute first.")
    print("Databricks mode enabled.")

print(f"Loaded config from: {cfg['_config_path']}")
print(f"RUN_MODE: {run_mode}")
print(f"Greeting: {cfg.get('greeting', 'Hello from Databricks Connect')}")


In [ ]:
greeting = str(cfg.get("greeting", "Hello from Databricks Connect"))
print(greeting)

who_df = spark.sql("select current_user() as user, current_timestamp() as ts")
_show_df(who_df, 10)

numbers_df = spark.range(5)
_show_df(numbers_df, 10)
